In [1]:
!pip uninstall -y datasets huggingface_hub
!pip install datasets==3.6.0 huggingface_hub==0.34.4
!pip install jsonlines

from datasets import load_dataset
import ast
import re
import json
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

Found existing installation: datasets 3.6.0
Uninstalling datasets-3.6.0:
  Successfully uninstalled datasets-3.6.0
Found existing installation: huggingface-hub 0.34.4
Uninstalling huggingface-hub-0.34.4:
  Successfully uninstalled huggingface-hub-0.34.4
  Using cached datasets-3.6.0-py3-none-any.whl.metadata (19 kB)
  Using cached huggingface_hub-0.34.4-py3-none-any.whl.metadata (14 kB)
Using cached datasets-3.6.0-py3-none-any.whl (491 kB)
Using cached huggingface_hub-0.34.4-py3-none-any.whl (561 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.0.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.34.4 which is incompatible.


In [2]:
# load dataset
dataset = load_dataset("lara-martin/FIREBALL")

print(dataset["train"][:5])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


{'speaker_id': [278369453363180276, 321444462285149813, 229748498274533401, 130221309971754716, 130221309971754716], 'before_utterances': [["It's been a long while since I've last HBed monsters from scratch, so please don't mind on field edits"], ['Move 35 feet then misty step 30 feet'], ["You're pulled 30 ft. towards the Swordmancer"], ['Just gonna gang on the Wanderer', 'Did they all approach me?'], ["Well gwm, didn't help me there"]], 'combat_state_before': [[{'name': 'Vivi', 'hp': '<48/48 HP; Healthy>', 'class': None, 'race': 'Displacer Beast Kitten', 'attacks': 'Tentacles', 'spells': '', 'actions': None, 'effects': '', 'description': None, 'controller_id': '278369453363180276'}, {'name': 'Nimue Willowleaf', 'hp': '<121/121 HP; Healthy>', 'class': 'Witch 17', 'race': 'Changeling', 'attacks': 'Unarmed Strike, Dagger, Invigorating Charm', 'spells': "Dispel Magic, Mending, Dimension Door, Vitriolic Sphere, Hold Monster, Conjure Elemental, Enervation, Acid Splash, Bestow Curse, Chill T

In [3]:
# helpers for labeling
def extract_modifier(cmd):
    tokens = cmd.strip().split()
    lt = [t.lower().strip() for t in tokens]

    if not lt:
        return None

    if lt[0] == "!cast":
        parts = []
        for tok in tokens[1:]:
            if tok.startswith("-") or tok.lower() in ["adv", "dis"]:
                break
            parts.append(tok)
        return " ".join(parts).strip('"').strip("'").lower() if parts else None

    if lt[0] in ["!a", "!attack"]:
        parts = []
        for tok in tokens[1:]:
            if tok.startswith("-") or tok.lower() in ["adv", "dis"]:
                break
            parts.append(tok)
        return " ".join(parts).strip('"').strip("'").lower() if parts else None

    if lt[0] == "!i" and len(lt) > 1:
        if lt[1] in ["cast", "c"]:
            parts = []
            for tok in tokens[2:]:
                if tok.startswith("-") or tok.lower() in ["adv", "dis"]:
                    break
                parts.append(tok)
            return " ".join(parts).strip('"').strip("'").lower() if parts else None

        if lt[1] in ["a", "attack"]:
            parts = []
            for tok in tokens[2:]:
                if tok.startswith("-") or tok.lower() in ["adv", "dis"]:
                    break
                parts.append(tok)
            return " ".join(parts).strip('"').strip("'").lower() if parts else None

    return None


def classify_action(command):
    cmd = command.lower().strip()

    if cmd.startswith("!cast") or cmd.startswith("!i cast") or cmd.startswith("!i c"):
        return "spell"

    if cmd.startswith("!a") or cmd.startswith("!attack") or cmd.startswith("!i a"):
        return "attack"

    return "other"


def extract_result_text(record):
    result_list = ast.literal_eval(record["automation_results"])
    if not result_list:
        return None
    return result_list[0].replace("\n", " ").strip()

In [4]:
# label data
rows = []

for i in range(len(dataset["train"])):
    cmd_list = ast.literal_eval(dataset["train"][i]["commands_norm"])

    if not cmd_list:
        continue

    cmd = cmd_list[0]
    prefix = cmd.split()[0].lower()

    if prefix.startswith("!m"):
        continue

    action = classify_action(cmd)

    if action == "other":
        continue

    modifier = extract_modifier(cmd)

    if modifier is None:
        continue

    text = extract_result_text(dataset["train"][i])

    if text is None:
        continue

    rows.append({
        "text": text,
        "action": action,
        "modifier": modifier,
        "command": cmd
    })

df = pd.DataFrame(rows)

print(df.head())
print(df["action"].value_counts())
print(df["modifier"].value_counts().head(30))

                                                text  action modifier  \
0  Nimue Willowleaf casts Mage Armor! Nimue Willo...   spell    armor   
1  Skylar attacks with a Rapier +3! Skylar attack...  attack   rapier   
2  The Wandering Swordmancer attacks with an Atta...  attack   attack   
3  Wanderer attacks with a Dawn's Strife! Wandere...  attack     dawn   
4  Wanderer attacks with a Dawn's Strife! Wandere...  attack     dawn   

                                             command  
0                                 !cast armor -t nim  
1                             !a rapier -t as4 -rr 2  
2       !i a attack -t sky -rr 2 -d1 4d8+1[slashing]  
3                                     !a dawn -t AS1  
4  !a dawn -t AS2 adv -d 10[slashing^] -b -5 -f "...  
action
attack    101227
spell      41348
Name: count, dtype: int64
modifier
bite              4109
rapier            1989
claw              1900
longbow           1813
unarmed           1377
longsword         1320
bless            

In [5]:
# logistic regression for action
X = df["text"]
y_action = df["action"]

vectorizer_action_lr = TfidfVectorizer(max_features=5000)
X_action = vectorizer_action_lr.fit_transform(X)

X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(
    X_action,
    y_action,
    test_size=0.2,
    random_state=42,
    stratify=y_action
)

model_action_lr = LogisticRegression(max_iter=1000)
model_action_lr.fit(X_train_a, y_train_a)

pred_action_lr = model_action_lr.predict(X_test_a)

print(f'Accuracy : {accuracy_score(y_test_a, pred_action_lr):.4f}')
print(f'Precision: {precision_score(y_test_a, pred_action_lr, average="weighted", zero_division=0):.4f}')
print(f'Recall   : {recall_score(y_test_a, pred_action_lr, average="weighted", zero_division=0):.4f}')
print(f'F1       : {f1_score(y_test_a, pred_action_lr, average="weighted", zero_division=0):.4f}')

Accuracy : 0.9942
Precision: 0.9942
Recall   : 0.9942
F1       : 0.9942


In [6]:
# ffnn
SEED = 42
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X.toarray(), dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class FFNNClassifier(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(DEVICE)
            logits = model(X_batch)
            preds = torch.argmax(logits, dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(y_batch.numpy())

    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
    rec = recall_score(all_labels, all_preds, average="weighted", zero_division=0)
    f1 = f1_score(all_labels, all_preds, average="weighted", zero_division=0)

    return acc, prec, rec, f1

In [15]:
# ffnn for action
X = df["text"].tolist()
y = df["action"].tolist()

label_encoder_action = LabelEncoder()
y_encoded = label_encoder_action.fit_transform(y)

X_train_texts, X_test_texts, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

vectorizer_action_nn = TfidfVectorizer(max_features=5000)
X_train = vectorizer_action_nn.fit_transform(X_train_texts).astype(np.float32)
X_test = vectorizer_action_nn.transform(X_test_texts).astype(np.float32)

train_dataset = TextDataset(X_train, y_train)
test_dataset = TextDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

model_action_nn = FFNNClassifier(
    input_dim=X_train.shape[1],
    num_classes=len(label_encoder_action.classes_),
    hidden_dim=256,
    dropout=0.3
).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_action_nn.parameters(), lr=1e-3)

for epoch in range(15):
    loss = train_epoch(model_action_nn, train_loader, optimizer, criterion)
    print(f"Epoch {epoch+1}: loss = {loss:.4f}")

acc, prec, rec, f1 = evaluate(model_action_nn, test_loader)

print(f'Accuracy : {acc:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall   : {rec:.4f}')
print(f'F1       : {f1:.4f}')

Epoch 1: loss = 0.0534
Epoch 2: loss = 0.0119
Epoch 3: loss = 0.0086
Epoch 4: loss = 0.0065
Epoch 5: loss = 0.0051
Epoch 6: loss = 0.0042
Epoch 7: loss = 0.0034
Epoch 8: loss = 0.0029
Epoch 9: loss = 0.0024
Epoch 10: loss = 0.0022
Epoch 11: loss = 0.0018
Epoch 12: loss = 0.0017
Epoch 13: loss = 0.0015
Epoch 14: loss = 0.0014
Epoch 15: loss = 0.0013
Accuracy : 0.9966
Precision: 0.9966
Recall   : 0.9966
F1       : 0.9966


In [8]:
# only modifiers that have more than 50 samples in the dataset are used
counts = df["modifier"].value_counts()
valid_modifiers = counts[counts >= 50].index

df_modifier = df[df["modifier"].isin(valid_modifiers)].copy()

print(df_modifier["modifier"].value_counts())

modifier
bite            4109
rapier          1989
claw            1900
longbow         1813
unarmed         1377
                ... 
spell             50
fate              50
divine favor      50
blunder           50
warning           50
Name: count, Length: 467, dtype: int64


In [9]:
# lr for modifier
X = df_modifier["text"]
y_modifier = df_modifier["modifier"]

vectorizer_modifier_lr = TfidfVectorizer(max_features=5000)
X_modifier = vectorizer_modifier_lr.fit_transform(X)

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_modifier,
    y_modifier,
    test_size=0.2,
    random_state=42,
    stratify=y_modifier
)

model_modifier_lr = LogisticRegression(max_iter=1000)
model_modifier_lr.fit(X_train_m, y_train_m)

pred_modifier_lr = model_modifier_lr.predict(X_test_m)

print(f'Accuracy : {accuracy_score(y_test_m, pred_modifier_lr):.4f}')
print(f'Precision: {precision_score(y_test_m, pred_modifier_lr, average="weighted", zero_division=0):.4f}')
print(f'Recall   : {recall_score(y_test_m, pred_modifier_lr, average="weighted", zero_division=0):.4f}')
print(f'F1       : {f1_score(y_test_m, pred_modifier_lr, average="weighted", zero_division=0):.4f}')

Accuracy : 0.8154
Precision: 0.8207
Recall   : 0.8154
F1       : 0.8025


In [10]:
# ffnn for modifier
X = df_modifier["text"].tolist()
y = df_modifier["modifier"].tolist()

label_encoder_modifier = LabelEncoder()
y_encoded = label_encoder_modifier.fit_transform(y)

X_train_texts, X_test_texts, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

vectorizer_modifier_nn = TfidfVectorizer(max_features=5000)
X_train = vectorizer_modifier_nn.fit_transform(X_train_texts).astype(np.float32)
X_test = vectorizer_modifier_nn.transform(X_test_texts).astype(np.float32)

train_dataset = TextDataset(X_train, y_train)
test_dataset = TextDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

model_modifier_nn = FFNNClassifier(
    input_dim=X_train.shape[1],
    num_classes=len(label_encoder_modifier.classes_),
    hidden_dim=256,
    dropout=0.3
).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_modifier_nn.parameters(), lr=1e-3)

for epoch in range(20):
    loss = train_epoch(model_modifier_nn, train_loader, optimizer, criterion)
    print(f"Epoch {epoch+1}: loss = {loss:.4f}")

acc, prec, rec, f1 = evaluate(model_modifier_nn, test_loader)

print(f'Accuracy : {acc:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall   : {rec:.4f}')
print(f'F1       : {f1:.4f}')

Epoch 1: loss = 2.9825
Epoch 2: loss = 0.8835
Epoch 3: loss = 0.5576
Epoch 4: loss = 0.4257
Epoch 5: loss = 0.3470
Epoch 6: loss = 0.2927
Epoch 7: loss = 0.2541
Epoch 8: loss = 0.2225
Epoch 9: loss = 0.1995
Epoch 10: loss = 0.1802
Epoch 11: loss = 0.1653
Epoch 12: loss = 0.1533
Epoch 13: loss = 0.1436
Epoch 14: loss = 0.1346
Epoch 15: loss = 0.1285
Epoch 16: loss = 0.1208
Epoch 17: loss = 0.1153
Epoch 18: loss = 0.1108
Epoch 19: loss = 0.1064
Epoch 20: loss = 0.1024
Accuracy : 0.8520
Precision: 0.8534
Recall   : 0.8520
F1       : 0.8493


In [11]:
# extract target from text
def getTarget(text, targets):
    tokens = re.findall(r"\b\w+\b", text.lower())

    for word in sorted(targets, key=len, reverse=True):
        if word.lower() in tokens:
            return word

    return "none"

In [12]:
# lr predictior
def predict_to_json_lr(utterance, targets):
    X_action = vectorizer_action_lr.transform([utterance])
    X_modifier = vectorizer_modifier_lr.transform([utterance])

    predicted_action = model_action_lr.predict(X_action)[0]
    predicted_modifier = model_modifier_lr.predict(X_modifier)[0]
    predicted_target = getTarget(utterance, targets)

    return {
        "action": predicted_action,
        "modifier": predicted_modifier,
        "target": predicted_target
    }

In [16]:
# ffnn predictor
def predict_to_json_nn(utterance, targets):
    X_action = vectorizer_action_nn.transform([utterance]).astype(np.float32)
    X_action_tensor = torch.tensor(X_action.toarray(), dtype=torch.float32).to(DEVICE)

    model_action_nn.eval()
    with torch.no_grad():
        logits = model_action_nn(X_action_tensor)
        pred_idx = torch.argmax(logits, dim=1).item()

    predicted_action = str(label_encoder_action.inverse_transform([pred_idx])[0])

    X_modifier = vectorizer_modifier_nn.transform([utterance]).astype(np.float32)
    X_modifier_tensor = torch.tensor(X_modifier.toarray(), dtype=torch.float32).to(DEVICE)

    model_modifier_nn.eval()
    with torch.no_grad():
        logits = model_modifier_nn(X_modifier_tensor)
        pred_idx = torch.argmax(logits, dim=1).item()

    predicted_modifier = str(label_encoder_modifier.inverse_transform([pred_idx])[0])
    predicted_target = getTarget(utterance, targets)

    return {
        "action": predicted_action,
        "modifier": predicted_modifier,
        "target": predicted_target
    }

In [17]:
# test
tests = [
    "I shoot john with my longbow",
    "I cast bless on james",
]

targets = {"john", "peter", "james"}

for t in tests:
    result = predict_to_json_nn(t, targets)

    print("Input:", t)
    print("Output:", result)

Input: I shoot john with my longbow
Output: {'action': 'attack', 'modifier': 'longbow', 'target': 'john'}
Input: I cast bless on james
Output: {'action': 'spell', 'modifier': 'bless', 'target': 'james'}
